In [ ]:
from pathlib import Path
import json
import csv
import pandas as pd

BASE_DIR = Path.cwd() / "Full_dataset"

REVIEWS_JSON = BASE_DIR / "Automotive_5.json" / "Automotive_5.json"
META_JSON    = BASE_DIR / "meta_Automotive.json" / "meta_Automotive.json"

REVIEWS_CSV = BASE_DIR / "automotive_reviews.csv"
META_CSV    = BASE_DIR / "automotive_meta.csv"

print("Reviews JSON exists:", REVIEWS_JSON.is_file())
print("Meta JSON exists:", META_JSON.is_file())

# From combined_meta.json
META_FIELDS = [
    'ASIN:', 'ASIN: ', 'Item model number:', 'Label:', 'Number of Discs:',
    'Original Release Date:', 'Shipping Weight:', 'also_buy', 'also_view',
    'asin', 'brand', 'category', 'date', 'description', 'details', 'feature',
    'fit', 'imageURL', 'imageURLHighRes', 'main_cat', 'price', 'rank',
    'similar_item', 'tech1', 'tech2', 'title'
]

# From combined_data.json
DATA_FIELDS = [
    'Color Name:', 'Color:', 'Format:', 'Item Package Quantity:',
    'Model Number:', 'Number of Items:', 'Package Quantity:',
    'Package Type:', 'Size:', 'Style Name:', 'Style:',
    'asin', 'image', 'overall', 'reviewText', 'reviewTime',
    'reviewerID', 'reviewerName', 'style', 'summary',
    'unixReviewTime', 'verified', 'vote'
]

def jsonlines_to_csv_with_known_fields(json_path: Path, csv_path: Path, fieldnames, log_every: int = 100_000):
    """
    Convert JSON Lines → CSV in a single pass, using a known list of fieldnames.
    - Assumes one JSON object per line (Amazon format).
    - Keeps only rows with a non-empty 'asin'.
    - Writes only the fields in `fieldnames` (missing fields become blank).
    """
    count = 0

    with json_path.open("r", encoding="utf-8") as fin, \
         csv_path.open("w", encoding="utf-8", newline="") as fout:

        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()

        for line_num, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            asin = obj.get("asin")
            if not asin:
                continue

            # Only keep the known fields; missing keys will be None
            row = {k: obj.get(k) for k in fieldnames}
            writer.writerow(row)
            count += 1

            if count % log_every == 0:
                print(f"{json_path.name}: wrote {count} rows")

    print(f"{json_path.name}: finished, total rows written = {count} → {csv_path.name}")

jsonlines_to_csv_with_known_fields(REVIEWS_JSON, REVIEWS_CSV, DATA_FIELDS)
jsonlines_to_csv_with_known_fields(META_JSON,    META_CSV,  META_FIELDS)

# Load both CSVs
reviews_df = pd.read_csv(REVIEWS_CSV)
meta_df    = pd.read_csv(META_CSV)

print("Reviews shape:", reviews_df.shape)
print("Meta shape:", meta_df.shape)

# Make sure asin is string in both
reviews_df["asin"] = reviews_df["asin"].astype(str)
meta_df["asin"]    = meta_df["asin"].astype(str)

# Inner join: only rows where asin exists in both
joined_df = reviews_df.merge(meta_df, on="asin", how="inner")
print("Joined shape:", joined_df.shape)

AUTO_JOINED_CSV = BASE_DIR / "automotive_joined_meta_reviews.csv"
joined_df.to_csv(AUTO_JOINED_CSV, index=False)
AUTO_JOINED_CSV

from pathlib import Path
import pandas as pd
import re, html

BASE_DIR = Path.cwd() / "Full_dataset"

REVIEWS_CSV = BASE_DIR / "automotive_reviews.csv"
META_CSV    = BASE_DIR / "automotive_meta.csv"

print("Reviews CSV exists:", REVIEWS_CSV.is_file())
print("Meta CSV exists:", META_CSV.is_file())

def strip_html(text):
    if not isinstance(text, str):
        return text
    # remove tags
    text = re.sub(r"<.*?>", "", text)
    # unescape HTML entities like &amp;
    return html.unescape(text).strip()

# Load meta
meta_df = pd.read_csv(META_CSV, low_memory=False)
print("Raw meta shape:", meta_df.shape)

# Ensure asin is string
meta_df["asin"] = meta_df["asin"].astype(str)

# Valid metadata rows
meta_valid_mask = meta_df["asin"].notna()
if "title" in meta_df.columns:
    meta_valid_mask &= meta_df["title"].notna()
if "main_cat" in meta_df.columns:
    meta_valid_mask &= meta_df["main_cat"].notna()

meta_df = meta_df[meta_valid_mask].copy()
print("After validity filter (meta):", meta_df.shape)

# Strip HTML from text fields
for col in ["title", "description"]:
    if col in meta_df.columns:
        meta_df[col] = meta_df[col].apply(strip_html)

# Deduplicate by asin (one row per product)
meta_df = meta_df.drop_duplicates(subset=["asin"], keep="last").reset_index(drop=True)
print("After dedupe by asin (meta):", meta_df.shape)

# Load reviews
reviews_df = pd.read_csv(REVIEWS_CSV, low_memory=False)
print("Raw reviews shape:", reviews_df.shape)

# Ensure asin is string
reviews_df["asin"] = reviews_df["asin"].astype(str)

# Valid review rows
reviews_valid_mask = reviews_df["asin"].notna()
if "reviewText" in reviews_df.columns:
    reviews_valid_mask &= reviews_df["reviewText"].notna()
if "overall" in reviews_df.columns:
    reviews_valid_mask &= reviews_df["overall"].notna()

reviews_df = reviews_df[reviews_valid_mask].copy()
print("After validity filter (reviews):", reviews_df.shape)

# Strip HTML from text fields
for col in ["reviewText", "summary"]:
    if col in reviews_df.columns:
        reviews_df[col] = reviews_df[col].apply(strip_html)

# Deduplicate reviews
dedupe_cols = [c for c in ["asin", "reviewerID", "unixReviewTime"] if c in reviews_df.columns]
if dedupe_cols:
    reviews_df = reviews_df.drop_duplicates(subset=dedupe_cols, keep="last")

print("After dedupe (reviews):", reviews_df.shape)

meta_asins    = set(meta_df["asin"])
review_asins  = set(reviews_df["asin"])
common_asins  = meta_asins & review_asins

print("Valid meta ASINs:", len(meta_asins))
print("Valid review ASINs:", len(review_asins))
print("Common ASINs:", len(common_asins))

# Filter both to ASIN intersection
meta_clean    = meta_df[meta_df["asin"].isin(common_asins)].copy()
reviews_clean = reviews_df[reviews_df["asin"].isin(common_asins)].copy()

print("Final meta shape:", meta_clean.shape)
print("Final reviews shape:", reviews_clean.shape)

AUTO_META_CLEAN_CSV    = BASE_DIR / "automotive_clean_meta.csv"
AUTO_REVIEWS_CLEAN_CSV = BASE_DIR / "automotive_clean_reviews.csv"

meta_clean.to_csv(AUTO_META_CLEAN_CSV, index=False)
reviews_clean.to_csv(AUTO_REVIEWS_CLEAN_CSV, index=False)

AUTO_META_CLEAN_CSV, AUTO_REVIEWS_CLEAN_CSV


from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd() / "Full_dataset"

META_CLEAN_PATH    = BASE_DIR / "Sports_and_Outdoors_meta.csv"
REVIEWS_CLEAN_PATH = BASE_DIR / "Sports_and_Outdoors_reviews.csv"

meta = pd.read_csv(META_CLEAN_PATH, low_memory=False)
reviews = pd.read_csv(REVIEWS_CLEAN_PATH, low_memory=False)

print("meta_clean shape:", meta.shape)
print("reviews_clean shape:", reviews.shape)

# Desired meta columns
desired_meta_cols = ["asin", "title", "brand", "price", "category", "description"]

meta_min = pd.DataFrame()
meta_min["asin"] = meta["asin"].astype(str)

if "title" in meta.columns:
    meta_min["title"] = meta["title"]
else:
    meta_min["title"] = pd.NA

if "brand" in meta.columns:
    meta_min["brand"] = meta["brand"]
else:
    meta_min["brand"] = pd.NA

if "price" in meta.columns:
    meta_min["price"] = meta["price"]
else:
    meta_min["price"] = pd.NA

# category: prefer 'category', fallback to 'main_cat'
if "category" in meta.columns:
    meta_min["category"] = meta["category"]
elif "main_cat" in meta.columns:
    meta_min["category"] = meta["main_cat"]
else:
    meta_min["category"] = pd.NA

if "description" in meta.columns:
    meta_min["description"] = meta["description"]
else:
    meta_min["description"] = pd.NA

meta_min = meta_min[desired_meta_cols]
print("minimal meta shape:", meta_min.shape)
meta_min.head()

desired_review_cols = ["reviewerID", "asin", "overall", "reviewText", "summary", "unixReviewTime"]

rev_min = pd.DataFrame()

# Ensure asin is string
if "asin" in reviews.columns:
    rev_min["asin"] = reviews["asin"].astype(str)
else:
    rev_min["asin"] = pd.NA

for col in ["reviewerID", "overall", "reviewText", "summary", "unixReviewTime"]:
    if col in reviews.columns:
        rev_min[col] = reviews[col]
    else:
        rev_min[col] = pd.NA

# Reorder columns to match desired schema
rev_min = rev_min[["reviewerID", "asin", "overall", "reviewText", "summary", "unixReviewTime"]]

print("minimal reviews shape:", rev_min.shape)
rev_min.head()

AUTO_META_MIN_PATH    = BASE_DIR / "Sports_and_Outdoors_final_meta.csv"
AUTO_REVIEWS_MIN_PATH = BASE_DIR / "Sports_and_Outdoors_final_reviews.csv"

meta_min.to_csv(AUTO_META_MIN_PATH, index=False)
rev_min.to_csv(AUTO_REVIEWS_MIN_PATH, index=False)

AUTO_META_MIN_PATH, AUTO_REVIEWS_MIN_PATH